## Instalación y importación




In [ ]:
!pip install transformers torch pyarrow mlflow -q

import pandas as pd
import numpy as np
import torch
import time
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import mlflow
import mlflow.sklearn
from transformers import pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score
)

print("MLflow version:", mlflow.__version__)
print("✅ Librerías listas")


## Carga de Datos de RAW

In [ ]:
# ── Cargar datos ──────────────────────────────────────────
df_train = pd.read_parquet('/content/sentiment_raw_train.parquet')
df_test  = pd.read_parquet('/content/sentiment_raw_test.parquet')

print(f"Train shape : {df_train.shape}")
print(f"Test shape  : {df_test.shape}")
print("\nColumnas:", df_train.columns.tolist())

## Cargar el pipeline de HuggingFace

In [ ]:
# ── Mapear etiquetas ──────────────────────────────────────
LABEL_MAP = {0: 'NEGATIVE', 1: 'POSITIVE', 4: 'POSITIVE'}
df_train['true_label'] = df_train['label'].map(LABEL_MAP)
df_test['true_label']  = df_test['label'].map(LABEL_MAP)
df_train = df_train.dropna(subset=['true_label'])
df_test  = df_test.dropna(subset=['true_label'])

# ── Pipeline HuggingFace ──────────────────────────────────
import torch, time
from tqdm.notebook import tqdm
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    revision="714eb0f",
    device=device
)

def run_inference(df, name, batch_size=64):
    texts = df['text'].astype(str).str[:512].tolist()
    total = len(texts)
    preds, scores = [], []

    print(f"\n🚀 {name} ({total:,} registros) | batch={batch_size}")
    start = time.time()

    # ── Batching manual → tqdm actualiza después de cada batch ──
    batches = [texts[i:i+batch_size] for i in range(0, total, batch_size)]

    with tqdm(total=total, desc=name, unit="tweet") as bar:
        for batch in batches:
            results = classifier(batch, truncation=True)
            for r in results:
                preds.append(r['label'])
                scores.append(r['score'])
            bar.update(len(batch))   # ← actualiza la barra después de cada batch

    latency = time.time() - start
    df['pred_label'] = preds
    df['score']      = scores

    print(f"✅ {name} | {latency:.1f}s ({latency/60:.1f} min) | "
          f"{total/latency:,.0f} t/s | {latency/total*1000:.3f} ms/tweet")

    return df, latency

# ── Ejecutar ──────────────────────────────────────────────
df_train, lat_train = run_inference(df_train, "TRAIN")
df_test,  lat_test  = run_inference(df_test,  "TEST")

df_train = df_train.dropna(subset=['true_label', 'pred_label'])
df_test  = df_test.dropna(subset=['true_label', 'pred_label'])

print(f"\nlat_train = {lat_train:.1f}s | lat_test = {lat_test:.1f}s")


## Métricas

In [ ]:
def get_metricas(df):
    y_true   = df['true_label']
    y_pred   = df['pred_label']
    prob_pos = df.apply(
        lambda r: r['score'] if r['pred_label'] == 'POSITIVE' else 1 - r['score'], axis=1
    )
    return {
        'accuracy'   : accuracy_score(y_true, y_pred),
        'f1_macro'   : f1_score(y_true, y_pred, average='macro'),
        'precision'  : precision_score(y_true, y_pred, average='macro'),
        'recall'     : recall_score(y_true, y_pred, average='macro'),
        'roc_auc'    : roc_auc_score((y_true == 'POSITIVE').astype(int), prob_pos)
    }

metrics_train = get_metricas(df_train)
metrics_test  = get_metricas(df_test)

for split_name, df, m in [('TRAIN', df_train, metrics_train),
                           ('TEST',  df_test,  metrics_test)]:
    print(f"\n{'='*55}")
    print(f"📊 RESULTADOS PARA: {split_name}")
    print(f"{'='*55}")
    print(f"  Accuracy  : {m['accuracy']:.4f}")
    print(f"  F1 Macro  : {m['f1_macro']:.4f}")
    print(f"  Precision : {m['precision']:.4f}")
    print(f"  Recall    : {m['recall']:.4f}")
    print(f"  ROC-AUC   : {m['roc_auc']:.4f}")
    print()
    print(classification_report(df['true_label'], df['pred_label'],
                                target_names=['NEGATIVE', 'POSITIVE']))


## Gráficas (Train vs Test)

In [ ]:
fig = plt.figure(figsize=(14, 16))
fig.suptitle('DistilBERT (SST-2) — Análisis de Sentimientos: Train vs Test',
             fontsize=15, fontweight='bold')

gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35,
                       height_ratios=[1, 1.1, 1.1])

# ── FILA 0: Barras de métricas (span completo) ────────────
ax_m = fig.add_subplot(gs[0, :])

metric_labels = ['accuracy', 'f1_macro', 'precision', 'recall', 'roc_auc']
vals_train    = [metrics_train[k] for k in metric_labels]
vals_test     = [metrics_test[k]  for k in metric_labels]

x     = np.arange(len(metric_labels))
width = 0.35
bars1 = ax_m.bar(x - width/2, vals_train, width, label='Train',
                  color='#5B9BD5', edgecolor='white')
bars2 = ax_m.bar(x + width/2, vals_test,  width, label='Test',
                  color='#ED7D31', edgecolor='white')

ax_m.set_xticks(x)
ax_m.set_xticklabels(metric_labels, fontsize=11)
ax_m.set_ylim(0, 1.13)
ax_m.set_ylabel('Score'); ax_m.legend(fontsize=10)
ax_m.set_title('Resumen de métricas: Train / Test — DistilBERT (SST-2)',
               fontsize=12, fontweight='bold')
ax_m.grid(axis='y', linestyle='--', alpha=0.4)
ax_m.set_facecolor('#F9F9F9')
for bar, val in zip(list(bars1) + list(bars2), vals_train + vals_test):
    ax_m.text(bar.get_x() + bar.get_width()/2, val + 0.01,
              f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

# ── FILAS 1 y 2: Confusión + Confidence Score ─────────────
for row, (split_name, df) in enumerate([('TRAIN', df_train), ('TEST', df_test)]):
    y_true = df['true_label'].tolist()
    y_pred = df['pred_label'].tolist()

    # Matriz de Confusión
    ax_cm = fig.add_subplot(gs[row + 1, 0])
    cm = confusion_matrix(y_true, y_pred, labels=['NEGATIVE', 'POSITIVE'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax_cm,
                xticklabels=['NEG', 'POS'], yticklabels=['NEG', 'POS'],
                annot_kws={"size": 14})
    ax_cm.set_title(f'{split_name} — Matriz de Confusión', fontweight='bold')
    ax_cm.set_ylabel('Real'); ax_cm.set_xlabel('Predicho')

    # Confidence Score
    ax_sc = fig.add_subplot(gs[row + 1, 1])
    for lbl, color in zip(['POSITIVE', 'NEGATIVE'], ['#2ecc71', '#e74c3c']):
        subset = df[df['pred_label'] == lbl]['score']
        ax_sc.hist(subset, bins=40, alpha=0.6, label=lbl, color=color, edgecolor='white')
    ax_sc.axvline(df['score'].mean(), color='navy', linestyle='--',
                  label=f"Media: {df['score'].mean():.3f}")
    ax_sc.set_title(f'{split_name} — Confidence Score', fontweight='bold')
    ax_sc.set_xlabel('Score'); ax_sc.set_ylabel('Frecuencia')
    ax_sc.legend()

plt.savefig('05_distilbert_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Guardado: 05_distilbert_dashboard.png")


## Registro en MLflow

In [ ]:
# ── Guardar la gráfica y predicciones como artefactos ─────
df_test[['text','true_label','pred_label','score']].to_csv(
    'distilbert_test_predictions.csv', index=False
)

# ── Conectar al servidor EC2-A ────────────────────────────
mlflow.set_tracking_uri("http://ec2-52-5-36-177.compute-1.amazonaws.com:5000")
mlflow.set_experiment("Parcial_1_NLP")

with mlflow.start_run(run_name="HuggingFace_DistilBERT_SST2_Reference") as run:

    # ── Tags ─────────────────────────────────────────────
    mlflow.set_tags({
        "user"          : "alan osorio",
        "model_type"    : "DistilBERT",
        "architecture"  : "Transformer",
        "encoding"      : "Tokenizer (WordPiece)",
        "dataset"       : "Sentiment140Twitter",
        "notebook"      : "05_distilbert.ipynb",
        "experiment_type": "D_comparison_huggingface",
        "note"          : "SST-2 model NOT trained on Twitter — reference baseline only"
    })

    # ── Parámetros del modelo ─────────────────────────────
    mlflow.log_params({
        "model_name"    : "distilbert-base-uncased-finetuned-sst-2-english",
        "batch_size"    : 128,
        "truncation"    : True,
        "max_length"    : 512,
        "device"        : "GPU" if device == 0 else "CPU",
        "pretrained_on" : "SST-2 (movie reviews)",
        "n_train"       : len(df_train),
        "n_test"        : len(df_test),
    })

    # ── Métricas en TRAIN ─────────────────────────────────
    mlflow.log_metrics({
        "train_accuracy"  : round(metrics_train['accuracy'],  4),
        "train_f1_macro"  : round(metrics_train['f1_macro'],  4),
        "train_precision" : round(metrics_train['precision'], 4),
        "train_recall"    : round(metrics_train['recall'],    4),
        "train_roc_auc"   : round(metrics_train['roc_auc'],   4),
    })

    # ── Métricas en TEST ──────────────────────────────────
    mlflow.log_metrics({
        "test_accuracy"   : round(metrics_test['accuracy'],  4),
        "test_f1_macro"   : round(metrics_test['f1_macro'],  4),
        "test_precision"  : round(metrics_test['precision'], 4),
        "test_recall"     : round(metrics_test['recall'],    4),
        "test_roc_auc"    : round(metrics_test['roc_auc'],   4),
    })

    # ── Latencia (requisito del PDF) ──────────────────────
    mlflow.log_metrics({
        "latency_train_sec"      : round(lat_train, 2),
        "latency_test_sec"       : round(lat_test, 2),
        "latency_ms_per_sample"  : round(lat_test / len(df_test) * 1000, 4),
    })

    # ── Artefactos ────────────────────────────────────────
    mlflow.log_artifact("05_distilbert_dashboard.png")
    mlflow.log_artifact("distilbert_test_predictions.csv")

    print("=" * 55)
    print("  ✅ Run registrado exitosamente en MLflow")
    print(f"  Servidor     : http://ec2-52-5-36-177.compute-1.amazonaws.com:5000")
    print(f"  Experimento  : Parcial_1_NLP")
    print(f"  Corrida      : HuggingFace_DistilBERT_SST2_Reference")
    print(f"  Run ID       : {run.info.run_id}")
    print(f"  Test F1      : {metrics_test['f1_macro']:.4f}")
    print(f"  Test AUC     : {metrics_test['roc_auc']:.4f}")
    print(f"  Latencia Test: {lat_test:.2f}s")
    print("=" * 55)
